In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
from scipy.stats import entropy
from scipy.optimize import curve_fit
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. 境界特定シミュレーション設定
# ==========================================
N_TRIALS = 800
# D=10から100まで10刻みでスキャンし、絶壁化の閾値を特定
DIMENSIONS = np.arange(10, 110, 10)
SAMPLE_SIZES = np.logspace(3.5, 6.0, num=35, dtype=int)
EPSILON_THRESHOLD = 0.001

output_dir = "hypothesis3_brittleness_boundary"
os.makedirs(output_dir, exist_ok=True)

print("相転移境界（絶壁化の閾値）の高解像度スキャンを開始します...")

# ==========================================
# 2. 単一試行関数 (H3統合DGP)
# ==========================================
def simulate_h3_single_trial(D, N):
    K = 5
    cell_raw = np.random.uniform(0, 0.2, D)
    P_cell = cell_raw / np.sum(cell_raw)

    betas = 10 ** np.random.uniform(-1, 1.5, D)
    X_choice = np.random.uniform(0, 0.2, (D, K))

    P_joint = np.zeros((D, K))
    for d in range(D):
        exp_val = np.exp(betas[d] * X_choice[d])
        P_choice_given_d = exp_val / np.sum(exp_val)
        P_joint[d, :] = P_cell[d] * P_choice_given_d

    P_flat = P_joint.flatten()
    C_flat = np.random.multinomial(N, P_flat)
    Q_flat = C_flat / N
    Q_flat = np.where(Q_flat == 0, 1e-10, Q_flat)

    kl_div = entropy(P_flat, Q_flat)
    return 1 if kl_div > EPSILON_THRESHOLD else 0

# ==========================================
# 3. グリッド探索とカーブフィッティング
# ==========================================
def sigmoid(x, kappa, x50):
    z = np.clip(kappa * (x - x50), -100, 100)
    return 1 / (1 + np.exp(z))

fit_results = []

for D in DIMENSIONS:
    print(f"次元数 D = {D} のスキャン中...")
    x_data = []
    y_data = []

    for N_samp in SAMPLE_SIZES:
        results = Parallel(n_jobs=-1)(delayed(simulate_h3_single_trial)(D, N_samp) for _ in range(N_TRIALS))
        rate = sum(results) / N_TRIALS
        x_data.append(np.log10(N_samp))
        y_data.append(rate)

    try:
        # 上限を緩和してより正確な鋭さを捉える
        p0 = [20.0, np.mean(x_data)]
        popt, _ = curve_fit(sigmoid, x_data, y_data, p0=p0, bounds=([0.1, 3.0], [100.0, 7.0]))
        kappa, x50 = popt
        delta_x = (2 * np.log(19)) / kappa
        # サンプルサイズの倍率に変換 (10^delta_x)
        sample_multiplier = 10 ** delta_x

        fit_results.append({
            "Dimension_D": D,
            "Acceleration_Kappa": kappa,
            "Transition_Width_Log10": delta_x,
            "Sample_Size_Multiplier": sample_multiplier
        })
    except Exception as e:
        print(f"D={D} fit error: {e}")

df_fit = pd.DataFrame(fit_results)
df_fit.to_csv(f"{output_dir}/brittleness_boundary_metrics.csv", index=False)
display(df_fit)

# ==========================================
# 4. 可視化
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = 'tab:red'
ax1.set_xlabel('Categorical Dimension D', fontsize=12)
ax1.set_ylabel('Acceleration Parameter ($\kappa$)', color=color1, fontsize=12)
ax1.plot(df_fit["Dimension_D"], df_fit["Acceleration_Kappa"], marker='o', color=color1, linewidth=2, label="Collapse Steepness ($\kappa$)")
ax1.tick_params(axis='y', labelcolor=color1)
ax1.axvline(x=40, color='gray', linestyle='--', alpha=0.7, label='Critical Threshold ($D \approx 40$)')

ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('Transition Sample Multiplier (100% $\to$ 5% Failure)', color=color2, fontsize=12)
ax2.plot(df_fit["Dimension_D"], df_fit["Sample_Size_Multiplier"], marker='s', color=color2, linewidth=2, label="Sample Multiplier Needed")
ax2.tick_params(axis='y', labelcolor=color2)

plt.title("Identification of the Phase Transition Boundary (Brittleness Threshold)", fontsize=14)
fig.tight_layout()
plt.savefig(f"{output_dir}/brittleness_boundary.png", dpi=300)
plt.close()

# ==========================================
# 5. ZIP化とダウンロード
# ==========================================
zip_filename = "hypothesis3_brittleness_boundary.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"\n境界スキャンが完了しました。結果を圧縮しました: {zip_filename}")
files.download(zip_filename)

<>:103: SyntaxWarning: invalid escape sequence '\k'
<>:104: SyntaxWarning: invalid escape sequence '\k'
<>:103: SyntaxWarning: invalid escape sequence '\k'
<>:104: SyntaxWarning: invalid escape sequence '\k'
/tmp/ipykernel_1265/3651846914.py:103: SyntaxWarning: invalid escape sequence '\k'
  ax1.set_ylabel('Acceleration Parameter ($\kappa$)', color=color1, fontsize=12)
/tmp/ipykernel_1265/3651846914.py:104: SyntaxWarning: invalid escape sequence '\k'
  ax1.plot(df_fit["Dimension_D"], df_fit["Acceleration_Kappa"], marker='o', color=color1, linewidth=2, label="Collapse Steepness ($\kappa$)")


相転移境界（絶壁化の閾値）の高解像度スキャンを開始します...
次元数 D = 10 のスキャン中...
次元数 D = 20 のスキャン中...
次元数 D = 30 のスキャン中...
次元数 D = 40 のスキャン中...
次元数 D = 50 のスキャン中...
次元数 D = 60 のスキャン中...
次元数 D = 70 のスキャン中...
次元数 D = 80 のスキャン中...
次元数 D = 90 のスキャン中...
次元数 D = 100 のスキャン中...


,Dimension_D,Acceleration_Kappa,Transition_Width_Log10,Sample_Size_Multiplier
0,10,17.807435,0.330698,2.141400
1,20,27.219671,0.216346,1.645684
2,30,31.521940,0.186818,1.537512
3,40,35.134335,0.167610,1.470992
4,50,39.812122,0.147917,1.405778
5,60,45.773364,0.128653,1.344785
6,70,44.812589,0.131411,1.353353
7,80,50.408430,0.116823,1.308649
8,90,49.972099,0.117843,1.311727
9,100,57.150593,0.103041,1.267773



境界スキャンが完了しました。結果を圧縮しました: hypothesis3_brittleness_boundary.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>